# Implementacion Algoritmo - Greedy

In [3]:
import time
import heapq
from itertools import product, permutations

In [9]:
class Tarea:
    def __init__(self, id, tiempo, prioridad, dependencias):
        self.id = id
        self.tiempo = tiempo
        self.prioridad = prioridad
        self.dependencias = dependencias

    def __repr__(self):
        return str(self.id)

In [10]:
def asignar_tareas(tareas, num_servidores, criterio="prioridad"):
    # Cada servidor: (momento en que queda libre, id, lista de tareas asignadas)
    servidores = [(0, i, []) for i in range(num_servidores)]
    heapq.heapify(servidores)

    fin_tarea = {}              # NUEVO: cuándo termina cada tarea
    pendientes = list(tareas)   # NUEVO: tareas que faltan por asignar

    while pendientes:
        # NUEVO (cambio 2): tareas cuyas dependencias ya fueron asignadas
        disponibles = [t for t in pendientes
                       if all(dep in fin_tarea for dep in t.dependencias)]

        # Decisión greedy: la mejor según el criterio
        tarea = max(disponibles, key=lambda t: getattr(t, criterio))

        # Servidor que se libera primero (igual que tu código)
        carga, id_servidor, lista = heapq.heappop(servidores)

        # NUEVO (cambio 3): esperar a que terminen las dependencias
        fin_deps = max((fin_tarea[dep] for dep in tarea.dependencias), default=0)
        inicio = max(carga, fin_deps)
        fin = inicio + tarea.tiempo

        lista.append(tarea)
        fin_tarea[tarea.id] = fin
        heapq.heappush(servidores, (fin, id_servidor, lista))
        pendientes.remove(tarea)

    # NUEVO (cambio 4): tiempo total y asignación ordenada por servidor
    tiempo_total = max(carga for carga, _, _ in servidores)
    asignacion = {id_s: lista for _, id_s, lista in sorted(servidores, key=lambda s: s[1])}
    return tiempo_total, asignacion

# Pruebas

In [13]:
servidores_pequeño = 2
tareas_pequeño = [
    Tarea("A", 5, 2, []),
    Tarea("B", 3, 1, ["A"]),
    Tarea("C", 4, 3, ["A"]),
    Tarea("D", 2, 1, ["B", "C"]),
]


In [14]:
servidores_mediano = 3
tareas_mediano = [
    Tarea("B", 13, 1, []),
    Tarea("C", 5, 3, ["A"]),
    Tarea("A", 8, 2, []),
    Tarea("D", 5, 2, ["A"]),
    Tarea("E", 4, 1, ["B", "C"]),
    Tarea("F", 6, 2, ["C"]),
    Tarea("G", 8, 2, ["E", "A"]),
    Tarea("H", 10, 2, ["D", "E", "F"]),
    Tarea("I", 7, 3, ["G"]),
]

In [15]:
servidores_grande = 3
tareas_grande = [
    Tarea("B", 13, 1, []),
    Tarea("C", 5, 3, ["A"]),
    Tarea("A", 8, 2, []),
    Tarea("D", 5, 2, ["A"]),
    Tarea("E", 4, 1, ["B", "C"]),
    Tarea("F", 6, 2, ["C"]),
    Tarea("G", 8, 2, ["E", "A"]),
    Tarea("H", 10, 2, ["D", "E", "F"]),
    Tarea("I", 7, 3, ["G"]),
    Tarea("J", 9, 1, ["B"]),
]

In [22]:
servidores_xl = 2
tareas_xl = [
    Tarea("B", 13, 1, []),
    Tarea("C", 5, 3, ["A"]),
    Tarea("A", 8, 2, []),
    Tarea("D", 5, 2, ["A"]),
    Tarea("E", 4, 1, ["B", "C"]),
    Tarea("F", 6, 2, ["C"]),
    Tarea("G", 8, 2, ["E", "A"]),
    Tarea("H", 10, 2, ["D", "E", "F"]),
    Tarea("I", 7, 3, ["G"]),
    Tarea("J", 9, 1, ["B"]),
    Tarea("K", 6, 2, []),
    Tarea("L", 11, 3, ["K"]),
    Tarea("M", 4, 1, ["J", "K"]),
    Tarea("N", 7, 2, ["H"]),
    Tarea("O", 3, 3, ["L", "F"]),
    Tarea("P", 12, 1, ["M"]),
    Tarea("Q", 5, 2, ["I", "O"]),
    Tarea("R", 9, 3, ["N", "P"]),
    Tarea("S", 2, 1, ["Q"]),
    Tarea("T", 8, 2, ["R", "S"]),
]

In [25]:
casos = [
    ("Pequeño", tareas_pequeño, servidores_pequeño),
    ("Mediano", tareas_mediano, servidores_mediano),
    ("Grande", tareas_grande, servidores_grande),
    ("Muy Grande", tareas_xl, servidores_xl),
]

for nombre, tareas, servidores in casos:
    inicio = time.perf_counter()
    resultado = asignar_tareas(tareas, servidores)
    fin = time.perf_counter()
    print(f"{nombre}: {resultado}")
    print(f"Tiempo: {fin - inicio:.7f} segundos\n")

Pequeño: (11, {0: [A, B, D], 1: [C]})
Tiempo: 0.0000399 segundos

Mediano: (45, {0: [A, F, G], 1: [C, B, I], 2: [D, E, H]})
Tiempo: 0.0000520 segundos

Grande: (47, {0: [A, F, G, J], 1: [C, B, I], 2: [D, E, H]})
Tiempo: 0.0000478 segundos

Muy Grande: (112, {0: [A, D, F, L, B, G, H, Q, J, P, S, T], 1: [C, K, O, E, I, N, M, R]})
Tiempo: 0.0001133 segundos

